# Uncertainty is not one number

## Estimation uncertainty, model-selection instability and economic decisions

Our previous notebooks addressed the structure of predictors, regression inference, predictive accuracy, missing observations and cost-sensitive selection.

Here we address a different question: when two statistically reasonable models suggest different interventions, *how confident should we be in the decision?*

We deliberately distinguish:
1. Variation in full-model OLS coefficients across bootstrap resamples.
2. Variation in predictions from a fixed estimator.
3. Variation from OOB-driven changes in which estimator is chosen.
4. Irreducible observation noise.
5. Decision regret, intervention rate and the frequency of mistaken interventions.

These are different statistical objects. Selection frequencies are not posterior model probabilities. Bootstrap predictive intervals are not automatically calibrated frequentist confidence intervals.

## Methodology and validation

Each bootstrap replicate draws a sample of training rows with replacement. A model is fitted on those rows, but its selection score is evaluated strictly on *out-of-bag* observations, whose original indices never appear in the bootstrap sample.

The candidate algorithms have prespecified regularisation parameters so duplicated bootstrap rows cannot contaminate an internal cross-validation split. Four policies are evaluated:

- Fixed OLS.
- Fixed ridge regression.
- Best OOB prediction MSE.
- Best OOB economic utility averaged over a cost grid.

Selected models produce out-of-sample predictions from the fitted in-bag estimator. Their selection probabilities and prediction distributions can therefore be analysed without touching test outcomes.

The independent test set is used only for final coverage and economic evaluation. The synthetic conditional mean is known by construction, but is never used during estimation or selection.

In [ ]:
from __future__ import annotations

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from uncertainty_analysis import run_uncertainty_study

BOOTSTRAPS: int = int(os.environ.get("UNCERTAINTY_BOOTSTRAPS", "60"))
N_TRAIN: int = int(os.environ.get("UNCERTAINTY_N_TRAIN", "180"))
N_TEST: int = int(os.environ.get("UNCERTAINTY_N_TEST", "300"))
if BOOTSTRAPS < 6 or N_TRAIN < 64 or N_TEST < 10:
    raise ValueError("Invalid notebook experiment settings")
print(f"Bootstrap draws: {BOOTSTRAPS}, n_train: {N_TRAIN}, n_test: {N_TEST}")

## 1. Coefficient uncertainty is not model-selection uncertainty

In the complete-data scenario, compare analytic OLS coefficient intervals with nonparametric bootstrap percentile intervals. Neither automatically resolves model misspecification or post-selection inference.

Two strongly correlated coefficients can be individually unstable although their combined contribution to the response remains useful. The coefficient analysis is therefore restricted to a prespecified full-model OLS fit.

In [ ]:
study = run_uncertainty_study(
    scenario="low_variance_signal",
    stress="clean",
    n_train=N_TRAIN,
    n_test=N_TEST,
    bootstrap_replicates=BOOTSTRAPS,
    seed=2026,
)
display(study.coefficient_table.head(8).round(3))
print(f"Mean OOB size: {study.draws.oob_sizes.mean():.1f}")

## 2. The law of total variance under changing model labels

For bootstrap prediction \(P\) and chosen algorithm label \(M\), the exact finite-mixture identity is:

\[
\operatorname{Var}(P)
=\operatorname{E}\!\left[\operatorname{Var}(P\mid M)\right]
+\operatorname{Var}\!\left(\operatorname{E}[P\mid M]\right).
\]

Our module computes both terms at each independent test case. This is a **descriptive partition**, not proof that switching algorithms *caused* the between-label term. Bootstrap resampling, estimator fitting and the selection mechanism are statistically coupled.

Moreover, a bootstrap label-frequency distribution is not a calibrated probability over the true data-generating model.

In [ ]:
display(study.variance_table.round(4))
selection = pd.concat(
    [
        pd.Series(labels, name="selected_model").value_counts()
        .rename_axis("selected_model").reset_index(name="frequency")
        .assign(policy=policy)
        for policy, labels in study.draws.selected_models.items()
    ],
    ignore_index=True,
)
selection["proportion"] = selection["frequency"] / BOOTSTRAPS
display(selection.sort_values(["policy", "frequency"], ascending=[True, False]))

## 3. Conditional-mean uncertainty versus predictive uncertainty

The bootstrap distribution of fitted conditional benefits describes estimator variability. It does **not** contain irreducible observation noise.

For a separate illustrative observation interval, we add new Gaussian noise using the known simulation parameter \(\sigma=0.65\), never inferred from test outcomes. We then compare empirical 90% coverage for:

- The synthetic test conditional mean, using bootstrap 5%/95% predictive-mean quantiles.
- Independently generated test outcomes, using bootstrap predictions plus new synthetic observation noise.

Neither interval method has guaranteed nominal coverage. In particular, model selection complicates calibration.

In [ ]:
display(study.coverage_table.round(3))
fig, ax = plt.subplots(figsize=(8, 4))
cases = np.arange(min(40, N_TEST))
values = study.draws.predictions["Selected utility"][:, cases]
lower, upper = np.quantile(values, [0.05, 0.95], axis=0)
ax.plot(cases, values.mean(axis=0), label="Bootstrap mean")
ax.fill_between(cases, lower, upper, alpha=0.25, label="Bootstrap 90% central range")
ax.set(xlabel="Independent test-case index", ylabel="Predicted net benefit",
       title="Selection-aware variation in fitted conditional benefit")
ax.legend()
fig.tight_layout()
plt.show()

## 4. Decision rules and the cost of uncertainty

We compare three policies using the fitted bootstrap prediction distribution:

- **Ensemble mean:** act when the mean estimated benefit exceeds the cost.
- **Majority vote:** act if more than half of the bootstrap predictions exceed the cost.
- **Lower 10% bound:** act only when the 10th percentile of benefit predictions exceeds the cost.

The third is a conservative screening rule, not a guarantee of safety or optimal loss. The outcome noise may differ from the uncertainty in the estimated conditional mean, and the cost of missing an opportunity can exceed that of an unnecessary action.

For each realised cost, measure regret against the known full-information oracle, overall action rate and the proportion of unprofitable actions and missed profitable opportunities.

In [ ]:
display(study.decision_table.round(3))
focus = study.decision_table.loc[
    study.decision_table["estimator"] == "Selected utility"
].sort_values("cost")
fig, ax = plt.subplots(figsize=(8, 4))
for rule, part in focus.groupby("rule", sort=True):
    ax.plot(part["cost"], part["regret"], marker="o", label=rule)
ax.set(xlabel="Known intervention cost", ylabel="Expected regret",
       title="Decision uncertainty and economic loss")
ax.legend()
fig.tight_layout()
plt.show()

## 5. Incomplete predictors: one controlled sensitivity analysis

Repeat the analysis after generating MAR missingness. The available pipelines impute missing predictors within each bootstrap training set, but no claim is made that median imputation identifies a full-information conditional mean under arbitrary MAR.

The oracle still has access to the original complete predictors. Its regret is consequently a measure of both model error and the irreducible information gap. Interpretation should emphasise comparisons *within* this stress condition, not absolute regret versus the clean model.

In [ ]:
stressed = run_uncertainty_study(
    scenario="low_variance_signal",
    stress="mar",
    n_train=N_TRAIN,
    n_test=N_TEST,
    bootstrap_replicates=BOOTSTRAPS,
    seed=2026,
)
display(stressed.variance_table.round(4))
display(stressed.coverage_table.round(3))
display(
    stressed.decision_table.loc[
        stressed.decision_table["estimator"] == "Selected utility"
    ].round(3)
)

## 6. Interpretation and limitations

These experiments answer separate questions rather than creating a single feature-importance score.

- A tight regression coefficient interval measures one form of parameter uncertainty, conditional on the linear model.
- An unstable choice of algorithm quantifies sensitivity to sampling and out-of-bag selection.
- The total predictive variance partition tells us how variation is distributed *conditional on observed selection labels*, not how much variance selection causally creates.
- A 90% bootstrap range is not necessarily calibrated to contain the conditional mean 90% of the time.
- Observation prediction intervals must incorporate outcome noise as well as uncertainty in fitted conditional means.
- Choosing an intervention from these scores is not equivalent to identifying a causal treatment effect.

### Next steps

Increase independent Monte Carlo replicates and calibrate empirical coverage; try conformal residual-based intervals with a genuine held-out calibration set, investigate heterogeneous treatment costs and policy errors, and stress the bootstrap under covariate shift, response outliers and MNAR scenarios.

We should only transfer this to a real decision-making dataset after defining an explicit outcome, economic utility and—if an intervention is proposed—a plausible identification strategy.